# Phase 4 — Knee Inference & Submission (self-contained)

Run on Kaggle (GPU notebook) with **Internet OFF**.
Attach: competition dataset + `knee-weights` + `dinov2-weights`.

Output: `/kaggle/working/submission.csv`


In [ ]:
import os, sys

os.makedirs("/kaggle/working/knee", exist_ok=True)
os.makedirs("/kaggle/working/knee/dinov2", exist_ok=True)
os.makedirs("/kaggle/working/knee/dinov2/layers", exist_ok=True)

# __init__.py
with open("/kaggle/working/knee/__init__.py", "w") as f:
    f.write('"\"\"\"knee package.\"\"\"\n__version__ = \"0.2.0\"\n')

# constants.py
with open("/kaggle/working/knee/constants.py", "w", encoding="utf-8") as f:
    f.write(
        'from typing import List\n'
        'STUDY_ID = "StudyInstanceUID"\n'
        'LABELS: List[str] = [\n'
        '    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",\n'
        '    "Medial OA", "Lateral OA", "PF OA",\n'
        '    "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture",\n'
        ']\n'
        'NUM_LABELS = len(LABELS)\n'
    )

print("package skeleton written")


In [ ]:
# --- config.py (embedded from repo) ---
_PAYLOAD_config_py = '"""Centralized config loader for the knee package.\n\nReads config.yaml from repo root (or /kaggle/working on Kaggle).\nAll modules should import config instead of hardcoding values.\n\nUsage:\n    from knee.config import CFG\n    data_dir = CFG["paths"]["local_data"]\n"""\n\nimport os\nfrom typing import Any, Dict\n\nimport yaml\n\n_REPO_ROOT = os.path.normpath(os.path.join(os.path.dirname(__file__), "..", ".."))\n_CFG: Dict[str, Any] = {}\n\n\ndef _find_config() -> str:\n    """Locate config.yaml: repo root first, then /kaggle/working."""\n    candidates = [\n        os.path.join(_REPO_ROOT, "config.yaml"),\n        "/kaggle/working/config.yaml",\n    ]\n    for p in candidates:\n        if os.path.isfile(p):\n            return p\n    raise FileNotFoundError(\n        f"config.yaml not found in {_REPO_ROOT} or /kaggle/working"\n    )\n\n\ndef load_config(path: str = None) -> Dict[str, Any]:\n    """Load and cache the YAML config. Repeated calls return the same dict."""\n    global _CFG\n    if _CFG:\n        return _CFG\n    if path is None:\n        path = _find_config()\n    with open(path, encoding="utf-8") as f:\n        _CFG = yaml.safe_load(f)\n    return _CFG\n\n\ndef get(*keys: str, default: Any = None) -> Any:\n    """Nested key access: get(\'paths\', \'local_data\')."""\n    cfg = load_config()\n    for k in keys:\n        if isinstance(cfg, dict):\n            cfg = cfg.get(k, default)\n        else:\n            return default\n    return cfg\n\n\n# convenience: load on import\nCFG = load_config\n'

_target = "/kaggle/working/knee/config.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_PAYLOAD_config_py)
import py_compile
py_compile.compile(_target, doraise=True)
print("config.py written:", len(_PAYLOAD_config_py), "chars")


In [ ]:
# --- constants.py (embedded from repo) ---
_PAYLOAD_constants_py = '"""Shared constants for the knee package."""\n\nfrom typing import List\n\nSTUDY_ID = "StudyInstanceUID"\n\nLABELS: List[str] = [\n    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",\n    "Medial OA", "Lateral OA", "PF OA",\n    "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture",\n]\n\nNUM_LABELS = len(LABELS)  # 12\n'

_target = "/kaggle/working/knee/constants.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_PAYLOAD_constants_py)
import py_compile
py_compile.compile(_target, doraise=True)
print("constants.py written:", len(_PAYLOAD_constants_py), "chars")


In [ ]:
# --- preprocess.py (embedded from repo) ---
_PAYLOAD_preprocess_py = '"""Shared preprocessing logic for the knee challenge (single source of truth).\n\nUsed by:\n- notebooks/10-preprocess.py   (Kaggle one-pass: train corpus -> npz shards)\n- notebooks/40-infer.py        (test-time preprocessing, efficiency-tuned)\n- local synthetic tests\n\nConstants (from Phase 0 recon):\n- 3 series per study by priority: sag-fluid > cor-fluid > ax-fluid,\n  fallbacks sag-nonfluid > cor-nonfluid > ax-nonfluid  (100% coverage verified)\n- 8 slices per series, uniformly sampled, sorted by ImagePositionPatient[2]\n  (100% present in recon; InstanceNumber fallback)\n- 224x224 uint8 (percentile-normalized per volume: p1/p99.5 clip -> scale)\n- per-volume normalization handles the mixed uint16/int16, 256-1024 matrices\n"""\n\nimport os\nimport re\nfrom typing import Dict, List, Optional, Tuple\n\nimport numpy as np\n\ntry:\n    import pydicom\nexcept ImportError as e:  # pragma: no cover\n    raise ImportError("pydicom required") from e\n\n# --- load from config.yaml (with hardcoded fallbacks for Kaggle/embedded use) ---\ntry:\n    from knee.config import get as cfg\n    IMG_SIZE = cfg("preprocessing", "img_size", default=224)\n    NUM_SLICES = cfg("preprocessing", "num_slices", default=8)\n    MAX_SERIES = cfg("preprocessing", "max_series", default=3)\n    P_LO = cfg("preprocessing", "percentile_low", default=1.0)\n    P_HI = cfg("preprocessing", "percentile_high", default=99.5)\n    _SERIES_PRIORITY = [\n        tuple(p) for p in cfg("preprocessing", "series_priority", default=[\n            ["Sagittal", 1], ["Coronal", 1], ["Axial", 1],\n            ["Sagittal", 0], ["Coronal", 0], ["Axial", 0],\n        ])\n    ]\nexcept Exception:\n    IMG_SIZE = 224\n    NUM_SLICES = 8\n    MAX_SERIES = 3\n    P_LO = 1.0\n    P_HI = 99.5\n    _SERIES_PRIORITY = [\n        ("Sagittal", 1), ("Coronal", 1), ("Axial", 1),\n        ("Sagittal", 0), ("Coronal", 0), ("Axial", 0),\n    ]\n\n\ndef find_data_dir(base: str = None) -> str:\n    """Locate the competition dir; handles nested /kaggle/input/competitions/<slug>/.\n\n    Bounded-depth walk; prunes train_series/test_series DICOM trees.\n    """\n    if base is None:\n        try:\n            from knee.config import get as cfg\n            base = cfg("paths", "kaggle_input", default="/kaggle/input")\n        except Exception:\n            base = "/kaggle/input"\n    if not os.path.isdir(base):\n        raise FileNotFoundError(f"{base} does not exist — attach the competition dataset")\n    for root, dirs, files in os.walk(base):\n        if "train.csv" in files or "test.csv" in files:\n            return root\n        depth = root[len(base):].count(os.sep) + (1 if root != base else 0)\n        if depth >= 3:\n            dirs[:] = []\n        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]\n    raise FileNotFoundError("competition data not found under " + base)\n\n\ndef select_series(study_series_rows, max_series: int = MAX_SERIES) -> List[str]:\n    """Pick up to max_series SeriesInstanceUIDs by plane/fluid priority.\n\n    `study_series_rows`: iterable of dicts/Series-like with keys\n    Anatomical_Plane, Fluid_Sensitive, SeriesInstanceUID.\n    Deterministic (sorted UIDs) so train/test selections match.\n    """\n    by_bucket: Dict[Tuple[str, int], List[str]] = {}\n    for row in study_series_rows:\n        plane = str(row["Anatomical_Plane"])\n        fluid = int(row["Fluid_Sensitive"])\n        by_bucket.setdefault((plane, fluid), []).append(str(row["SeriesInstanceUID"]))\n\n    chosen: List[str] = []\n    for bucket in _SERIES_PRIORITY:\n        cands = by_bucket.get(bucket, [])\n        if cands:\n            chosen.append(sorted(cands)[0])\n        if len(chosen) >= max_series:\n            break\n    # fill any remaining slots deterministically\n    for uid in sorted(u for cands in by_bucket.values() for u in cands):\n        if uid not in chosen and len(chosen) < max_series:\n            chosen.append(uid)\n    return chosen\n\n\ndef _read_slice_position(path: str) -> float:\n    """Header-only read for physical Z position; ~1ms. Falls back to 0.0."""\n    try:\n        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)\n        ipp = getattr(ds, "ImagePositionPatient", None)\n        if ipp is not None and len(ipp) >= 3:\n            return float(ipp[2])\n        num = getattr(ds, "InstanceNumber", None)\n        if num is not None:\n            return float(num)\n    except Exception:\n        pass\n    return 0.0\n\n\ndef _slice_sort_key(path: str) -> Tuple[float, str]:\n    """Header-only read for position; ~1ms. Falls back to filename."""\n    return (_read_slice_position(path), path)\n\n\ndef _select_indices_by_position(\n    n: int, num_slices: int, positions: List[float]\n) -> List[int]:\n    """Select slice indices with uniform physical spacing.\n\n    Samples evenly across the physical Z-axis span instead of file-count,\n    giving better anatomical coverage when slices are non-uniformly spaced.\n\n    Falls back to index-based uniform sampling when positions are unavailable\n    (all zeros) or when the series has fewer slices than requested.\n    """\n    if n <= num_slices:\n        return list(range(n))\n\n    # check if physical positions are meaningful\n    span = positions[-1] - positions[0]\n    if abs(span) < 1e-6:\n        # no physical spacing info — fall back to index-based\n        return sorted(set(int(i) for i in np.linspace(0, n - 1, num_slices)))\n\n    # sample uniformly across physical span\n    start, stop = positions[0], positions[-1]\n    targets = np.linspace(start, stop, num_slices)\n\n    idx = []\n    j = 0\n    for t in targets:\n        # advance pointer to nearest unchosen slice\n        best = j\n        best_dist = abs(positions[j] - t)\n        while j < n - 1:\n            d = abs(positions[j + 1] - t)\n            if d < best_dist:\n                j += 1\n                best = j\n                best_dist = d\n            else:\n                break\n        if best not in idx:\n            idx.append(best)\n        else:\n            # if duplicate, try neighbours\n            for offset in range(1, n):\n                for cand in (best - offset, best + offset):\n                    if 0 <= cand < n and cand not in idx:\n                        idx.append(cand)\n                        break\n                if len(idx) > len(targets):\n                    break\n\n    # pad if dedup shrank below num_slices\n    j = 0\n    while len(idx) < num_slices and j < n:\n        if j not in idx:\n            idx.append(j)\n        j += 1\n\n    return sorted(idx[:num_slices])\n\n\n_DCM_RE = re.compile(r"\\.dcm$", re.IGNORECASE)\n\n\ndef read_series(\n    series_dir: str,\n    num_slices: int = NUM_SLICES,\n    img_size: int = IMG_SIZE,\n    header_only_first: bool = False,\n) -> Optional[np.ndarray]:\n    """Read one DICOM series -> (num_slices, img_size, img_size) uint8.\n\n    Efficiency: header-scans ALL files ONCE (cheap) to sort + collect physical\n    positions, then pixel-decodes ONLY the sampled slices.\n\n    Returns None on unreadable/empty series (caller zero-fills).\n    """\n    try:\n        files = [os.path.join(series_dir, f) for f in os.listdir(series_dir)\n                 if _DCM_RE.search(f)]\n    except OSError:\n        return None\n    if not files:\n        return None\n\n    # --- single-pass header scan: read position + sort ---\n    entries = []  # [(position, path)]\n    for p in files:\n        entries.append((_read_slice_position(p), p))\n    entries.sort(key=lambda e: e[0])\n\n    n = len(entries)\n    positions = [e[0] for e in entries]\n    sorted_files = [e[1] for e in entries]\n    idx = _select_indices_by_position(n, num_slices, positions)\n\n    # --- pixel-decode only the selected slices ---\n    slices = []\n    for i in idx:\n        try:\n            ds = pydicom.dcmread(sorted_files[i], force=True)\n            img = ds.pixel_array.astype(np.float32)\n            slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\n            inter = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\n            if slope != 1.0 or inter != 0.0:\n                img = img * slope + inter\n            slices.append(img)\n        except Exception:\n            continue\n    if not slices:\n        return None\n\n    # pad to num_slices by repeating last slice\n    while len(slices) < num_slices:\n        slices.append(slices[-1])\n\n    vol = np.stack(slices)  # (S, H, W) float32\n\n    # per-volume percentile normalization -> uint8\n    finite = vol[np.isfinite(vol)]\n    if finite.size == 0:\n        return None\n    p_lo, p_hi = np.percentile(finite, [P_LO, P_HI])\n    if p_hi <= p_lo:\n        p_hi = p_lo + 1e-6\n    vol = np.clip(vol, p_lo, p_hi)\n    vol = (vol - p_lo) / (p_hi - p_lo)\n\n    # resize\n    try:\n        import cv2\n        out = np.empty((vol.shape[0], img_size, img_size), dtype=np.uint8)\n        for k in range(vol.shape[0]):\n            r = cv2.resize(vol[k], (img_size, img_size), interpolation=cv2.INTER_AREA)\n            out[k] = np.clip(np.rint(r * 255.0), 0, 255).astype(np.uint8)\n        return out\n    except ImportError:\n        # PIL fallback (Kaggle always has cv2, local may not)\n        from PIL import Image\n        out = np.empty((vol.shape[0], img_size, img_size), dtype=np.uint8)\n        for k in range(vol.shape[0]):\n            im = Image.fromarray((vol[k] * 255).astype(np.uint8))\n            out[k] = np.array(im.resize((img_size, img_size), Image.BILINEAR))\n        return out\n\n\ndef study_tensor(\n    study_dir: str,\n    series_uids: List[str],\n    num_slices: int = NUM_SLICES,\n    img_size: int = IMG_SIZE,\n) -> np.ndarray:\n    """Stack selected series -> (MAX_SERIES, NUM_SLICES, H, W) uint8.\n\n    Missing/unreadable series are zero-filled so the shape is fixed.\n    """\n    out = np.zeros((MAX_SERIES, num_slices, img_size, img_size), dtype=np.uint8)\n    for k, suid in enumerate(series_uids[:MAX_SERIES]):\n        sdir = os.path.join(study_dir, suid)\n        t = read_series(sdir, num_slices, img_size)\n        if t is not None:\n            out[k] = t\n    return out\n\n\ndef process_study(args):\n    """Multiprocessing worker: (uid, series_uids, series_root) -> result tuple.\n\n    Kept at module scope (picklable). Returns\n    (uid, ok, tensor) — tensor is zero-filled (MAX_SERIES, NUM_SLICES, H, W)\n    on failure so downstream stacking never breaks.\n    """\n    uid, series_uids, series_root = args\n    try:\n        t = study_tensor(os.path.join(series_root, uid), series_uids)\n        return uid, True, t\n    except Exception:\n        return uid, False, np.zeros(\n            (MAX_SERIES, NUM_SLICES, IMG_SIZE, IMG_SIZE), np.uint8\n        )\n'

_target = "/kaggle/working/knee/preprocess.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_PAYLOAD_preprocess_py)
import py_compile
py_compile.compile(_target, doraise=True)
print("preprocess.py written:", len(_PAYLOAD_preprocess_py), "chars")


In [ ]:
# --- inference.py (embedded from repo) ---
_PAYLOAD_inference_py = '"""Inference pipeline for the knee challenge.\n\nUsage:\n    from knee.inference import load_checkpoint, predict, write_submission\n\n    model, cfg = load_checkpoint("best_model.pt")\n    probs = predict(model, tensors, batch_size=32, device="cuda")\n    write_submission(probs, study_uids, "submission.csv")\n"""\n\nimport os\nfrom typing import List, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\n\nfrom knee.constants import LABELS\n\n\ndef load_checkpoint(path: str, device: str = None) -> Tuple[nn.Module, dict]:\n    """Load a checkpoint saved by Person A and return (model, config).\n\n    Checkpoint format (saved by A):\n        torch.save({\n            "model_state": model.module.state_dict(),\n            "config": {"img_size": 224, "num_slices": 8, "max_series": 3},\n            "label_order": constants.LABELS,\n            ...\n        })\n    """\n    if device is None:\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n\n    ckpt = torch.load(path, map_location=device, weights_only=False)\n    cfg = ckpt.get("config", {})\n\n    # Build model arch — import here so device is known\n    from knee.model import KneeModel\n    # KneeModel(backbone=None, num_labels=12): backbone auto-resolved via\n    # knee.dinov2 or torch.hub; config dict is only used by dataset.prep_tensor.\n    model = KneeModel()\n    model.load_state_dict(ckpt["model_state"], strict=True)\n    model.to(device)\n    model.eval()\n    return model, cfg\n\n\n@torch.no_grad()\ndef predict(\n    model: nn.Module,\n    tensors: np.ndarray,\n    batch_size: int = 32,\n    device: str = None,\n) -> np.ndarray:\n    """Run inference on preprocessed tensors.\n\n    Args:\n        model: loaded KneeModel\n        tensors: (N, 24, 3, 224, 224) float32 (after prep_tensor)\n        batch_size: inference batch size\n        device: "cuda" or "cpu"\n\n    Returns:\n        probs: (N, 12) float32 in [0, 1]\n    """\n    if device is None:\n        device = next(model.parameters()).device\n\n    N = tensors.shape[0]\n    all_probs = np.zeros((N, len(LABELS)), dtype=np.float32)\n\n    for start in range(0, N, batch_size):\n        end = min(start + batch_size, N)\n        batch = torch.from_numpy(tensors[start:end]).to(device)\n        logits = model(batch)  # (B, 12)\n        probs = torch.sigmoid(logits).cpu().numpy()\n        all_probs[start:end] = probs\n\n    return all_probs\n\n\ndef write_submission(\n    probs: np.ndarray,\n    study_uids: List[str],\n    out_path: str,\n    label_order: List[str] = None,\n) -> str:\n    """Write submission.csv in the exact format expected by Kaggle.\n\n    Args:\n        probs: (N, 12) probabilities\n        study_uids: list of StudyInstanceUID, length N\n        out_path: output CSV path\n        label_order: column order (defaults to knee.constants.LABELS)\n\n    Returns:\n        path to the written file\n    """\n    if label_order is None:\n        label_order = LABELS\n\n    assert len(probs) == len(study_uids), \\\n        f"probs ({len(probs)}) and uids ({len(study_uids)}) must match"\n    assert probs.shape[1] == len(label_order), \\\n        f"probs has {probs.shape[1]} columns, expected {len(label_order)}"\n\n    # Clip to [0, 1] and replace NaN\n    probs = np.clip(probs, 0.0, 1.0)\n    probs = np.nan_to_num(probs, nan=0.5)\n\n    df = pd.DataFrame(probs, columns=label_order)\n    df.insert(0, "StudyInstanceUID", study_uids)\n    df.to_csv(out_path, index=False)\n    return out_path\n'

_target = "/kaggle/working/knee/inference.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_PAYLOAD_inference_py)
import py_compile
py_compile.compile(_target, doraise=True)
print("inference.py written:", len(_PAYLOAD_inference_py), "chars")


In [ ]:
# --- config.yaml (embedded from repo) ---
_PAYLOAD_config_yaml = 'seed: 42\n\npaths:\n  local_data: "E:\\\\KneeAbnormal"\n  kaggle_input: "/kaggle/input"\n  kaggle_working: "/kaggle/working"\n  output_subdir: "shards"\n\npreprocessing:\n  img_size: 224\n  num_slices: 8\n  max_series: 3\n  shards_per_file: 2000\n  n_workers: 4\n  chunksize: 8\n  percentile_low: 1.0\n  percentile_high: 99.5\n  series_priority:\n    - ["Sagittal", 1]\n    - ["Coronal", 1]\n    - ["Axial", 1]\n    - ["Sagittal", 0]\n    - ["Coronal", 0]\n    - ["Axial", 0]\n\nlabels:\n  - "ACL"\n  - "MCL"\n  - "Medial Meniscus"\n  - "Lateral Meniscus"\n  - "Medial OA"\n  - "Lateral OA"\n  - "PF OA"\n  - "Effusion"\n  - "Synovitis"\n  - "Baker\'s"\n  - "Contusion"\n  - "Fracture"\n\nml:\n  tfidf_analyzer: "char_wb"\n  tfidf_ngram_range: [2, 5]\n  tfidf_min_df: 1\n  tfidf_max_features: 200000\n  tfidf_sublinear_tf: true\n  lr_C: 2.0\n  lr_max_iter: 3000\n  lr_class_weight: "balanced"\n\npseudo_labels:\n  positive_threshold: 0.75\n  negative_threshold: 0.25\n  ensemble_labels:\n    - "Medial Meniscus"\n    - "Medial OA"\n    - "Lateral OA"\n    - "Synovitis"\n    - "Fracture"\n'

_target = "/kaggle/working/knee/config.yaml"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_PAYLOAD_config_yaml)
import py_compile
py_compile.compile(_target, doraise=True)
print("config.yaml written:", len(_PAYLOAD_config_yaml), "chars")


In [ ]:
# --- knee/dinov2/vision_transformer.py ---
_DINOV2_dinov2_vision_transformer_py = '# Vendored from facebookresearch/dinov2 (Apache 2.0)\n# Minimal: inference-only, no xformers/torchvision dependency\n\nfrom functools import partial\nimport math\nimport logging\nfrom typing import Sequence, Tuple, Union, Callable\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.nn.init import trunc_normal_\n\nfrom knee.dinov2.layers import Mlp, PatchEmbed, SwiGLUFFNFused, MemEffAttention, NestedTensorBlock as Block\n\nlogger = logging.getLogger("dinov2")\n\n\ndef named_apply(fn: Callable, module: nn.Module, name="", depth_first=True, include_root=False) -> nn.Module:\n    if not depth_first and include_root:\n        fn(module=module, name=name)\n    for child_name, child_module in module.named_children():\n        child_name = ".".join((name, child_name)) if name else child_name\n        named_apply(fn=fn, module=child_module, name=child_name, depth_first=depth_first, include_root=True)\n    if depth_first and include_root:\n        fn(module=module, name=name)\n    return module\n\n\nclass DinoVisionTransformer(nn.Module):\n    def __init__(\n        self,\n        img_size=224,\n        patch_size=16,\n        in_chans=3,\n        embed_dim=768,\n        depth=12,\n        num_heads=12,\n        mlp_ratio=4.0,\n        qkv_bias=True,\n        ffn_bias=True,\n        proj_bias=True,\n        drop_path_rate=0.0,\n        drop_path_uniform=False,\n        init_values=None,\n        embed_layer=PatchEmbed,\n        act_layer=nn.GELU,\n        block_fn=Block,\n        ffn_layer="mlp",\n        block_chunks=1,\n        num_register_tokens=0,\n        interpolate_antialias=False,\n        interpolate_offset=0.1,\n        channel_adaptive=False,\n    ):\n        super().__init__()\n        norm_layer = partial(nn.LayerNorm, eps=1e-6)\n\n        self.num_features = self.embed_dim = embed_dim\n        self.num_tokens = 1\n        self.n_blocks = depth\n        self.num_heads = num_heads\n        self.patch_size = patch_size\n        self.num_register_tokens = num_register_tokens\n        self.interpolate_antialias = interpolate_antialias\n        self.interpolate_offset = interpolate_offset\n        self.bag_of_channels = channel_adaptive\n\n        self.patch_embed = embed_layer(img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)\n        num_patches = self.patch_embed.num_patches\n\n        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))\n        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + self.num_tokens, embed_dim))\n        assert num_register_tokens >= 0\n        self.register_tokens = (\n            nn.Parameter(torch.zeros(1, num_register_tokens, embed_dim)) if num_register_tokens else None\n        )\n\n        if drop_path_uniform is True:\n            dpr = [drop_path_rate] * depth\n        else:\n            dpr = np.linspace(0, drop_path_rate, depth).tolist()\n\n        if ffn_layer == "mlp":\n            ffn_layer = Mlp\n        elif ffn_layer == "swiglufused" or ffn_layer == "swiglu":\n            ffn_layer = SwiGLUFFNFused\n        elif ffn_layer == "identity":\n            def f(*args, **kwargs):\n                return nn.Identity()\n            ffn_layer = f\n        else:\n            raise NotImplementedError\n\n        blocks_list = [\n            block_fn(\n                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,\n                qkv_bias=qkv_bias, proj_bias=proj_bias, ffn_bias=ffn_bias,\n                drop_path=dpr[i], norm_layer=norm_layer, act_layer=act_layer,\n                ffn_layer=ffn_layer, init_values=init_values,\n            )\n            for i in range(depth)\n        ]\n        if block_chunks > 0:\n            self.chunked_blocks = True\n            chunked_blocks = []\n            chunksize = depth // block_chunks\n            for i in range(0, depth, chunksize):\n                chunked_blocks.append([nn.Identity()] * i + blocks_list[i:i + chunksize])\n            self.blocks = nn.ModuleList([BlockChunk(p) for p in chunked_blocks])\n        else:\n            self.chunked_blocks = False\n            self.blocks = nn.ModuleList(blocks_list)\n\n        self.norm = norm_layer(embed_dim)\n        self.head = nn.Identity()\n        self.mask_token = nn.Parameter(torch.zeros(1, embed_dim))\n        self.init_weights()\n\n    def init_weights(self):\n        trunc_normal_(self.pos_embed, std=0.02)\n        nn.init.normal_(self.cls_token, std=1e-6)\n        if self.register_tokens is not None:\n            nn.init.normal_(self.register_tokens, std=1e-6)\n        named_apply(init_weights_vit_timm, self)\n\n    def interpolate_pos_encoding(self, x, w, h):\n        previous_dtype = x.dtype\n        npatch = x.shape[1] - 1\n        N = self.pos_embed.shape[1] - 1\n        if npatch == N and w == h:\n            return self.pos_embed\n        pos_embed = self.pos_embed.float()\n        class_pos_embed = pos_embed[:, 0]\n        patch_pos_embed = pos_embed[:, 1:]\n        dim = x.shape[-1]\n        w0 = w // self.patch_size\n        h0 = h // self.patch_size\n        M = int(math.sqrt(N))\n        assert N == M * M\n        kwargs = {}\n        if self.interpolate_offset:\n            sx = float(w0 + self.interpolate_offset) / M\n            sy = float(h0 + self.interpolate_offset) / M\n            kwargs["scale_factor"] = (sx, sy)\n        else:\n            kwargs["size"] = (w0, h0)\n        patch_pos_embed = nn.functional.interpolate(\n            patch_pos_embed.reshape(1, M, M, dim).permute(0, 3, 1, 2),\n            mode="bicubic", antialias=self.interpolate_antialias, **kwargs,\n        )\n        assert (w0, h0) == patch_pos_embed.shape[-2:]\n        patch_pos_embed = patch_pos_embed.permute(0, 2, 3, 1).view(1, -1, dim)\n        return torch.cat((class_pos_embed.unsqueeze(0), patch_pos_embed), dim=1).to(previous_dtype)\n\n    def prepare_tokens_with_masks(self, x, masks=None):\n        B, nc, w, h = x.shape\n        x = self.patch_embed(x)\n        if masks is not None:\n            x = torch.where(masks.unsqueeze(-1), self.mask_token.to(x.dtype).unsqueeze(0), x)\n        x = torch.cat((self.cls_token.expand(x.shape[0], -1, -1), x), dim=1)\n        x = x + self.interpolate_pos_encoding(x, w, h)\n        if self.register_tokens is not None:\n            x = torch.cat((x[:, :1], self.register_tokens.expand(x.shape[0], -1, -1), x[:, 1:]), dim=1)\n        return x\n\n    def forward_features(self, x, masks=None):\n        if isinstance(x, list):\n            raise NotImplementedError("List input not supported in inference mode")\n        x = self.prepare_tokens_with_masks(x, masks)\n        for blk in self.blocks:\n            x = blk(x)\n        x_norm = self.norm(x)\n        return {\n            "x_norm_clstoken": x_norm[:, 0],\n            "x_norm_regtokens": x_norm[:, 1:self.num_register_tokens + 1],\n            "x_norm_patchtokens": x_norm[:, self.num_register_tokens + 1:],\n            "x_prenorm": x,\n            "masks": masks,\n        }\n\n    def forward(self, *args, is_training=False, **kwargs):\n        ret = self.forward_features(*args, **kwargs)\n        if is_training:\n            return ret\n        else:\n            return self.head(ret["x_norm_clstoken"])\n\n\nclass BlockChunk(nn.ModuleList):\n    def forward(self, x):\n        for b in self:\n            x = b(x)\n        return x\n\n\ndef init_weights_vit_timm(module: nn.Module, name: str = ""):\n    if isinstance(module, nn.Linear):\n        trunc_normal_(module.weight, std=0.02)\n        if module.bias is not None:\n            nn.init.zeros_(module.bias)\n\n\ndef vit_small(patch_size=14, num_register_tokens=0, in_chans=3, channel_adaptive=False,\n              block_chunks=0, init_values=1.0, **kwargs):\n    model = DinoVisionTransformer(\n        patch_size=patch_size,\n        embed_dim=384,\n        depth=12,\n        num_heads=6,\n        mlp_ratio=4,\n        block_fn=partial(Block, attn_class=MemEffAttention),\n        num_register_tokens=num_register_tokens,\n        in_chans=in_chans,\n        channel_adaptive=channel_adaptive,\n        block_chunks=block_chunks,\n        init_values=init_values,\n        **kwargs,\n    )\n    return model\n'

_target = "/kaggle/working/knee/dinov2/vision_transformer.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_vision_transformer_py)
print("dinov2/vision_transformer.py written")


In [ ]:
# --- knee/dinov2/__init__.py ---
_DINOV2_dinov2___init___py = '# Vendored DINOv2 from facebookresearch/dinov2 (Apache 2.0)\n# Minimal inference-only build — no xformers, no torchvision dependency\n\nimport math\nimport torch\nfrom knee.dinov2.vision_transformer import vit_small, DinoVisionTransformer\n\n__all__ = ["vit_small", "DinoVisionTransformer", "load_pretrained"]\n\n\ndef load_pretrained(model, pretrained_path, strict=False):\n    """Load pretrained DINOv2 weights with automatic pos_embed interpolation.\n\n    Handles the img_size mismatch: pretrained weights are from img_size=518\n    (1369 patches), but our model uses img_size=224 (256 patches).\n    The pos_embed is interpolated via bicubic to match.\n    """\n    sd = torch.load(pretrained_path, map_location="cpu", weights_only=True)\n\n    # Interpolate pos_embed if shape mismatches\n    if "pos_embed" in sd and "pos_embed" in model.state_dict():\n        target_shape = model.state_dict()["pos_embed"].shape\n        src_shape = sd["pos_embed"].shape\n        if src_shape != target_shape:\n            cls_token = sd["pos_embed"][:, :1]\n            patch_pos = sd["pos_embed"][:, 1:]\n            N_src = patch_pos.shape[1]\n            M_src = int(math.sqrt(N_src))\n            N_target = target_shape[1] - 1\n            M_target = int(math.sqrt(N_target))\n            dim = patch_pos.shape[2]\n            patch_pos_2d = patch_pos.reshape(1, M_src, M_src, dim).permute(0, 3, 1, 2)\n            patch_pos_interp = torch.nn.functional.interpolate(\n                patch_pos_2d, size=(M_target, M_target), mode="bicubic", antialias=True,\n            )\n            patch_pos_interp = patch_pos_interp.permute(0, 2, 3, 1).reshape(1, -1, dim)\n            sd["pos_embed"] = torch.cat([cls_token, patch_pos_interp], dim=1)\n\n    model.load_state_dict(sd, strict=strict)\n    return model\n'

_target = "/kaggle/working/knee/dinov2/__init__.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2___init___py)
print("dinov2/__init__.py written")


In [ ]:
# --- knee/dinov2/layers/attention.py ---
_DINOV2_dinov2_layers_attention_py = 'import logging\nimport torch\nfrom torch import nn, Tensor\n\nlogger = logging.getLogger("dinov2")\n\n\nclass Attention(nn.Module):\n    def __init__(\n        self,\n        dim: int,\n        num_heads: int = 8,\n        qkv_bias: bool = False,\n        proj_bias: bool = True,\n        attn_drop: float = 0.0,\n        proj_drop: float = 0.0,\n    ) -> None:\n        super().__init__()\n        self.dim = dim\n        self.num_heads = num_heads\n        head_dim = dim // num_heads\n        self.scale = head_dim ** -0.5\n\n        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)\n        self.attn_drop = attn_drop\n        self.proj = nn.Linear(dim, dim, bias=proj_bias)\n        self.proj_drop = nn.Dropout(proj_drop)\n\n    def forward(self, x: Tensor, is_causal: bool = False) -> Tensor:\n        B, N, C = x.shape\n        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)\n        q, k, v = torch.unbind(qkv, 2)\n        q, k, v = [t.transpose(1, 2) for t in [q, k, v]]\n        x = nn.functional.scaled_dot_product_attention(\n            q, k, v, attn_mask=None,\n            dropout_p=self.attn_drop if self.training else 0,\n            is_causal=is_causal,\n        )\n        x = x.transpose(1, 2).contiguous().view(B, N, C)\n        x = self.proj_drop(self.proj(x))\n        return x\n\n\nclass MemEffAttention(Attention):\n    """Memory-efficient attention using PyTorch SDPA (no xformers needed)."""\n    def forward(self, x: Tensor, attn_bias=None) -> Tensor:\n        if attn_bias is not None:\n            raise AssertionError("attn_bias not supported without xformers")\n        return super().forward(x)\n'

_target = "/kaggle/working/knee/dinov2/layers/attention.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_layers_attention_py)
print("dinov2/layers/attention.py written")


In [ ]:
# --- knee/dinov2/layers/block.py ---
_DINOV2_dinov2_layers_block_py = 'import logging\nfrom typing import Callable, List\nimport torch\nfrom torch import nn, Tensor\n\nfrom .attention import Attention, MemEffAttention\nfrom .drop_path import DropPath\nfrom .layer_scale import LayerScale\nfrom .mlp import Mlp\n\nlogger = logging.getLogger("dinov2")\n\n\nclass Block(nn.Module):\n    def __init__(\n        self,\n        dim: int,\n        num_heads: int,\n        mlp_ratio: float = 4.0,\n        qkv_bias: bool = False,\n        proj_bias: bool = True,\n        ffn_bias: bool = True,\n        drop: float = 0.0,\n        attn_drop: float = 0.0,\n        init_values=None,\n        drop_path: float = 0.0,\n        act_layer: Callable[..., nn.Module] = nn.GELU,\n        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,\n        attn_class: Callable[..., nn.Module] = Attention,\n        ffn_layer: Callable[..., nn.Module] = Mlp,\n    ) -> None:\n        super().__init__()\n        self.norm1 = norm_layer(dim)\n        self.attn = attn_class(\n            dim, num_heads=num_heads, qkv_bias=qkv_bias,\n            proj_bias=proj_bias, attn_drop=attn_drop, proj_drop=drop,\n        )\n        self.ls1 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()\n        self.drop_path1 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()\n\n        self.norm2 = norm_layer(dim)\n        mlp_hidden_dim = int(dim * mlp_ratio)\n        self.mlp = ffn_layer(\n            in_features=dim, hidden_features=mlp_hidden_dim,\n            act_layer=act_layer, drop=drop, bias=ffn_bias,\n        )\n        self.ls2 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()\n        self.drop_path2 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()\n        self.sample_drop_ratio = drop_path\n\n    def forward(self, x: Tensor) -> Tensor:\n        def attn_residual_func(x: Tensor) -> Tensor:\n            return self.ls1(self.attn(self.norm1(x)))\n\n        def ffn_residual_func(x: Tensor) -> Tensor:\n            return self.ls2(self.mlp(self.norm2(x)))\n\n        if self.training and self.sample_drop_ratio > 0.1:\n            from .block import drop_add_residual_stochastic_depth\n            x = drop_add_residual_stochastic_depth(x, residual_func=attn_residual_func, sample_drop_ratio=self.sample_drop_ratio)\n            x = drop_add_residual_stochastic_depth(x, residual_func=ffn_residual_func, sample_drop_ratio=self.sample_drop_ratio)\n        elif self.training and self.sample_drop_ratio > 0.0:\n            x = x + self.drop_path1(attn_residual_func(x))\n            x = x + self.drop_path1(ffn_residual_func(x))\n        else:\n            x = x + attn_residual_func(x)\n            x = x + ffn_residual_func(x)\n        return x\n\n\nclass NestedTensorBlock(Block):\n    def forward(self, x_or_x_list):\n        if isinstance(x_or_x_list, Tensor):\n            return super().forward(x_or_x_list)\n        elif isinstance(x_or_x_list, list):\n            raise AssertionError("Nested tensors require xformers")\n        else:\n            raise AssertionError\n'

_target = "/kaggle/working/knee/dinov2/layers/block.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_layers_block_py)
print("dinov2/layers/block.py written")


In [ ]:
# --- knee/dinov2/layers/drop_path.py ---
_DINOV2_dinov2_layers_drop_path_py = 'from torch import nn\n\n\ndef drop_path(x, drop_prob: float = 0.0, training: bool = False):\n    if drop_prob == 0.0 or not training:\n        return x\n    keep_prob = 1 - drop_prob\n    shape = (x.shape[0],) + (1,) * (x.ndim - 1)\n    random_tensor = x.new_empty(shape).bernoulli_(keep_prob)\n    if keep_prob > 0.0:\n        random_tensor.div_(keep_prob)\n    return x * random_tensor\n\n\nclass DropPath(nn.Module):\n    def __init__(self, drop_prob=None):\n        super().__init__()\n        self.drop_prob = drop_prob\n\n    def forward(self, x):\n        return drop_path(x, self.drop_prob, self.training)\n'

_target = "/kaggle/working/knee/dinov2/layers/drop_path.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_layers_drop_path_py)
print("dinov2/layers/drop_path.py written")


In [ ]:
# --- knee/dinov2/layers/layer_scale.py ---
_DINOV2_dinov2_layers_layer_scale_py = 'from typing import Optional, Union\nimport torch\nfrom torch import Tensor, nn\n\n\nclass LayerScale(nn.Module):\n    def __init__(\n        self,\n        dim: int,\n        init_values: Union[float, Tensor] = 1e-5,\n        inplace: bool = False,\n        device: Optional[torch.device] = None,\n        dtype: Optional[torch.dtype] = None,\n    ) -> None:\n        super().__init__()\n        self.inplace = inplace\n        self.init_values = init_values\n        self.gamma = nn.Parameter(torch.empty(dim, device=device, dtype=dtype))\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        nn.init.constant_(self.gamma, self.init_values)\n\n    def forward(self, x: Tensor) -> Tensor:\n        return x.mul_(self.gamma) if self.inplace else x * self.gamma\n'

_target = "/kaggle/working/knee/dinov2/layers/layer_scale.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_layers_layer_scale_py)
print("dinov2/layers/layer_scale.py written")


In [ ]:
# --- knee/dinov2/layers/mlp.py ---
_DINOV2_dinov2_layers_mlp_py = 'from typing import Callable, Optional\nfrom torch import Tensor, nn\n\n\nclass Mlp(nn.Module):\n    def __init__(\n        self,\n        in_features: int,\n        hidden_features: Optional[int] = None,\n        out_features: Optional[int] = None,\n        act_layer: Callable[..., nn.Module] = nn.GELU,\n        drop: float = 0.0,\n        bias: bool = True,\n    ) -> None:\n        super().__init__()\n        out_features = out_features or in_features\n        hidden_features = hidden_features or in_features\n        self.fc1 = nn.Linear(in_features, hidden_features, bias=bias)\n        self.act = act_layer()\n        self.fc2 = nn.Linear(hidden_features, out_features, bias=bias)\n        self.drop = nn.Dropout(drop)\n\n    def forward(self, x: Tensor) -> Tensor:\n        x = self.fc1(x)\n        x = self.act(x)\n        x = self.drop(x)\n        x = self.fc2(x)\n        x = self.drop(x)\n        return x\n'

_target = "/kaggle/working/knee/dinov2/layers/mlp.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_layers_mlp_py)
print("dinov2/layers/mlp.py written")


In [ ]:
# --- knee/dinov2/layers/patch_embed.py ---
_DINOV2_dinov2_layers_patch_embed_py = 'from typing import Callable, Optional, Tuple, Union\nfrom torch import Tensor\nimport torch.nn as nn\n\n\ndef make_2tuple(x):\n    if isinstance(x, tuple):\n        assert len(x) == 2\n        return x\n    assert isinstance(x, int)\n    return (x, x)\n\n\nclass PatchEmbed(nn.Module):\n    def __init__(\n        self,\n        img_size: Union[int, Tuple[int, int]] = 224,\n        patch_size: Union[int, Tuple[int, int]] = 16,\n        in_chans: int = 3,\n        embed_dim: int = 768,\n        norm_layer: Optional[Callable] = None,\n        flatten_embedding: bool = True,\n    ) -> None:\n        super().__init__()\n        image_HW = make_2tuple(img_size)\n        patch_HW = make_2tuple(patch_size)\n        patch_grid_size = (\n            image_HW[0] // patch_HW[0],\n            image_HW[1] // patch_HW[1],\n        )\n        self.img_size = image_HW\n        self.patch_size = patch_HW\n        self.patches_resolution = patch_grid_size\n        self.num_patches = patch_grid_size[0] * patch_grid_size[1]\n        self.in_chans = in_chans\n        self.embed_dim = embed_dim\n        self.flatten_embedding = flatten_embedding\n        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_HW, stride=patch_HW)\n        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()\n\n    def forward(self, x: Tensor) -> Tensor:\n        _, _, H, W = x.shape\n        patch_H, patch_W = self.patch_size\n        assert H % patch_H == 0, f"Input image height {H} is not a multiple of patch height {patch_H}"\n        assert W % patch_W == 0, f"Input image width {W} is not a multiple of patch width: {patch_W}"\n        x = self.proj(x)\n        H, W = x.size(2), x.size(3)\n        x = x.flatten(2).transpose(1, 2)\n        x = self.norm(x)\n        if not self.flatten_embedding:\n            x = x.reshape(-1, H, W, self.embed_dim)\n        return x\n'

_target = "/kaggle/working/knee/dinov2/layers/patch_embed.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_layers_patch_embed_py)
print("dinov2/layers/patch_embed.py written")


In [ ]:
# --- knee/dinov2/layers/swiglu_ffn.py ---
_DINOV2_dinov2_layers_swiglu_ffn_py = 'from typing import Callable, Optional\nfrom torch import Tensor, nn\nimport torch.nn.functional as F\n\n\nclass SwiGLUFFN(nn.Module):\n    def __init__(\n        self,\n        in_features: int,\n        hidden_features: Optional[int] = None,\n        out_features: Optional[int] = None,\n        act_layer: Callable[..., nn.Module] = None,\n        drop: float = 0.0,\n        bias: bool = True,\n    ) -> None:\n        super().__init__()\n        out_features = out_features or in_features\n        hidden_features = hidden_features or in_features\n        self.w12 = nn.Linear(in_features, 2 * hidden_features, bias=bias)\n        self.w3 = nn.Linear(hidden_features, out_features, bias=bias)\n\n    def forward(self, x: Tensor) -> Tensor:\n        x12 = self.w12(x)\n        x1, x2 = x12.chunk(2, dim=-1)\n        hidden = F.silu(x1) * x2\n        return self.w3(hidden)\n\n\nclass SwiGLUFFNFused(SwiGLUFFN):\n    def __init__(\n        self,\n        in_features: int,\n        hidden_features: Optional[int] = None,\n        out_features: Optional[int] = None,\n        act_layer: Callable[..., nn.Module] = None,\n        drop: float = 0.0,\n        bias: bool = True,\n    ) -> None:\n        out_features = out_features or in_features\n        hidden_features = hidden_features or in_features\n        hidden_features = (int(hidden_features * 2 / 3) + 7) // 8 * 8\n        super().__init__(\n            in_features=in_features, hidden_features=hidden_features,\n            out_features=out_features, bias=bias,\n        )\n'

_target = "/kaggle/working/knee/dinov2/layers/swiglu_ffn.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_layers_swiglu_ffn_py)
print("dinov2/layers/swiglu_ffn.py written")


In [ ]:
# --- knee/dinov2/layers/__init__.py ---
_DINOV2_dinov2_layers___init___py = '# Vendored from facebookresearch/dinov2 (Apache 2.0)\nfrom .mlp import Mlp\nfrom .patch_embed import PatchEmbed\nfrom .swiglu_ffn import SwiGLUFFNFused\nfrom .block import Block, NestedTensorBlock\nfrom .attention import Attention, MemEffAttention\n'

_target = "/kaggle/working/knee/dinov2/layers/__init__.py"
os.makedirs(os.path.dirname(_target), exist_ok=True)
with open(_target, "w", encoding="utf-8") as f:
    f.write(_DINOV2_dinov2_layers___init___py)
print("dinov2/layers/__init__.py written")


In [ ]:
# --- Inference payload ---
_INFER_PAYLOAD = '"""Phase 4 inference script — runs inside the self-contained Kaggle notebook.\n\nThis file is the PAYLOAD of notebooks/40-infer.ipynb.\nThe builder (build_infer_ipynb.py) embeds this verbatim.\n"""\n\nimport os\nimport sys\nimport time\nfrom multiprocessing import Pool\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, "/kaggle/working")\n\nfrom knee.config import get as cfg\nfrom knee.preprocess import (\n    IMG_SIZE, MAX_SERIES, NUM_SLICES,\n    find_data_dir, process_study, select_series,\n)\nfrom knee.dataset import prep_tensor\nfrom knee.model import KneeModel\nfrom knee.inference import predict, write_submission\nfrom knee.dinov2 import vit_small, load_pretrained\n\n\ndef find_checkpoint():\n    """Search for best_model.pt in /kaggle/input/."""\n    for root, dirs, files in os.walk("/kaggle/input"):\n        if "best_model.pt" in files:\n            return os.path.join(root, "best_model.pt")\n    raise FileNotFoundError("best_model.pt not found in /kaggle/input/")\n\n\ndef main():\n    t0 = time.time()\n\n    # 1. Find data\n    DATA_DIR = find_data_dir()\n    SERIES_ROOT = os.path.join(DATA_DIR, "test_series")\n    print("data dir:", DATA_DIR)\n\n    # 2. Read metadata\n    test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))\n    series_df = pd.read_csv(os.path.join(DATA_DIR, "test_series.csv"))\n    series_meta = series_df.to_dict("records")\n    study_series = series_df.groupby("StudyInstanceUID")\n\n    # 3. Select series per study\n    tasks = []\n    for uid in test_df["StudyInstanceUID"]:\n        if uid in study_series.groups:\n            rows = study_series.get_group(uid).to_dict("records")\n            uids = select_series(rows)\n        else:\n            uids = []\n        tasks.append((uid, uids, SERIES_ROOT))\n\n    print(f"test studies: {len(tasks)}")\n\n    # 4. Preprocess with multiprocessing\n    N_WORKERS = cfg("preprocessing", "n_workers", default=4)\n    CHUNKSIZE = cfg("preprocessing", "chunksize", default=8)\n\n    raw_tensors = {}\n    fails = 0\n    with Pool(N_WORKERS) as pool:\n        for uid, ok, tensor in pool.imap(process_study, tasks, chunksize=CHUNKSIZE):\n            if not ok:\n                fails += 1\n            raw_tensors[uid] = tensor\n\n    print(f"preprocessing done: {len(raw_tensors)} studies, {fails} failures, "\n          f"{time.time()-t0:.1f}s")\n\n    # 5. Apply prep_tensor (normalize, replicate channels)\n    uids_ordered = list(test_df["StudyInstanceUID"])\n    tensors = np.stack([raw_tensors[u] for u in uids_ordered])\n    tensors = prep_tensor(tensors)  # (N, 3, 8, 224, 224) uint8 -> (N, 24, 3, 224, 224) float32\n    print(f"prep_tensor done: {tensors.shape}, {tensors.dtype}")\n\n    # 6. Load model\n    ckpt_path = find_checkpoint()\n    print(f"checkpoint: {ckpt_path}")\n\n    from knee.inference import load_checkpoint\n    model, model_cfg = load_checkpoint(ckpt_path)\n    print(f"model loaded: {model_cfg}")\n\n    # 7. Predict\n    device = "cuda" if __import__("torch").cuda.is_available() else "cpu"\n    probs = predict(model, tensors, batch_size=32, device=device)\n    print(f"predictions: {probs.shape}, range [{probs.min():.4f}, {probs.max():.4f}]")\n\n    # 8. Write submission\n    out_path = "/kaggle/working/submission.csv"\n    write_submission(probs, uids_ordered, out_path)\n    print(f"submission written: {out_path}")\n    print(f"total time: {time.time()-t0:.1f}s")\n\n\nif __name__ == "__main__":\n    main()\n'

with open("/kaggle/working/40_infer_payload.py", "w", encoding="utf-8") as f:
    f.write(_INFER_PAYLOAD)

exec(open("/kaggle/working/40_infer_payload.py").read())
